# Notebook 08 — Optimization Experiments A, B, C

Three targeted experiments designed in response to the cross-docking finding
(notebook 07): TargetDiff generates structurally distinct pools for WT vs T790M,
but no selectivity differentiation emerged in cross-docking at 10 Å pocket radius.

| Experiment | Question | Runtime |
|------------|----------|---------|
| **A — Pocket Radius** | Does reducing the pocket from 10 Å → 6 Å amplify the T790M selectivity signal? | ~1.5 h (needs GPU for generation) |
| **B — Selectivity Outliers** | Which molecules already show the strongest selectivity, and what structural features explain it? | ~5 min (CPU only) |
| **C — ADMET Profiling** | Are the most selective candidates also drug-developable? | ~5 min (CPU only) |

**Recommended run order:** Start B + C first (CPU, instant results). Run A
separately on a GPU runtime. Results from all three feed into notebook 09.


In [ ]:
# -- Bootstrap: mount Drive, set paths, detect runtime --
import os, sys, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
except ImportError:
    PROJECT_ROOT = os.path.abspath('.')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=False)

try:
    import rdkit
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'rdkit', 'pandas', 'matplotlib', 'tqdm'], check=True)
    import rdkit

import torch
HAS_GPU = torch.cuda.is_available()
TARGETDIFF = os.path.join(PROJECT_ROOT, 'external', 'targetdiff')
CONDA_PY   = '/usr/local/bin/python3.11'   # conda env used by nb04/06/07

print('PROJECT_ROOT :', PROJECT_ROOT)
print('GPU available:', HAS_GPU,
      f'({torch.cuda.get_device_name(0)})' if HAS_GPU else '(CPU runtime — skip Section A)')
print('rdkit        :', rdkit.__version__)


---
## Section B — Selectivity Outlier Analysis
*CPU only · ~5 minutes*

Rank all 103 cross-docked molecules by **selectivity differential**
`Δ = vina_Mut − vina_WT`:
- **Δ < 0** → molecule binds WT *better* than T790M (WT-selective)
- **Δ > 0** → molecule binds T790M *better* than WT (Mut-selective)

We extract the top-5 of each class for structural analysis.


In [ ]:
# -- B1: Load combined_scores.csv and compute selectivity delta --
import pandas as pd
import numpy as np

scores = pd.read_csv(f'{PROJECT_ROOT}/results/vina_scores/combined_scores.csv')
print(f'Loaded {len(scores)} molecules  |  sources: {scores["source"].value_counts().to_dict()}')

# Selectivity: negative = WT-selective, positive = Mut-selective
scores['delta'] = scores['vina_Mut'] - scores['vina_WT']

# Separate generated mols from baselines
gen   = scores[scores['source'] != 'baseline'].copy()
bases = scores[scores['source'] == 'baseline'].copy()

print(f'\nGenerated pool stats (n={len(gen)}):')
print(f'  delta mean  : {gen["delta"].mean():.3f} kcal/mol')
print(f'  delta std   : {gen["delta"].std():.3f}')
print(f'  WT-selective (delta < -0.5) : {(gen["delta"] < -0.5).sum()}')
print(f'  Mut-selective (delta > 0.5) : {(gen["delta"] > 0.5).sum()}')
print(f'\nBaseline deltas:')
for _, row in bases.iterrows():
    label = row.get('name', row['source'])
    print(f'  {label:15s} delta={row["delta"]:+.3f}  (vina_WT={row["vina_WT"]:.2f}, vina_Mut={row["vina_Mut"]:.2f})')


In [ ]:
# -- B2: Top-5 WT-selective and top-5 Mut-selective molecules --
import pandas as pd

top_wt  = gen.nsmallest(5, 'delta')[['mol_id','source','smiles','vina_WT','vina_Mut','delta']]
top_mut = gen.nlargest(5,  'delta')[['mol_id','source','smiles','vina_WT','vina_Mut','delta']]

print('=== Top-5 WT-selective (delta most negative) ===')
print(top_wt.to_string(index=False))
print()
print('=== Top-5 Mut-selective (delta most positive) ===')
print(top_mut.to_string(index=False))

# Save for downstream use (Experiment C, Notebook 09)
os.makedirs(f'{PROJECT_ROOT}/results/analysis', exist_ok=True)
top_wt.to_csv(f'{PROJECT_ROOT}/results/analysis/top5_wt_selective.csv', index=False)
top_mut.to_csv(f'{PROJECT_ROOT}/results/analysis/top5_mut_selective.csv', index=False)
print('\nSaved to results/analysis/')


In [ ]:
# -- B3: Structure grid — top-5 WT-selective + top-5 Mut-selective --
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

def draw_grid(df, title, sort_col='delta', n=5):
    mols = [Chem.MolFromSmiles(s) for s in df['smiles']]
    legends = [
        f'delta={row["delta"]:+.2f}\nvina_WT={row["vina_WT"]:.1f}\nvina_Mut={row["vina_Mut"]:.1f}'
        for _, row in df.iterrows()
    ]
    img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(280, 220),
                                legends=legends)
    print(f'--- {title} ---')
    display(img)
    return img

img_wt  = draw_grid(top_wt,  'Top-5 WT-selective  (vina_WT << vina_Mut)')
img_mut = draw_grid(top_mut, 'Top-5 Mut-selective (vina_Mut << vina_WT)')


In [ ]:
# -- B4: Selectivity delta distribution histogram --
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: delta histogram by source pool
for src, color, label in [
    ('WT_generated',  '#1f77b4', 'WT-generated (n=50)'),
    ('Mut_generated', '#d62728', 'Mut-generated (n=50)'),
]:
    sub = gen[gen['source'] == src]['delta']
    axes[0].hist(sub, bins=15, alpha=0.55, color=color, label=label)

# Mark baseline deltas
for _, row in bases.iterrows():
    name = row.get('name', row['source'])
    axes[0].axvline(row['delta'], linestyle='--', linewidth=1.5,
                    label=f'{name} ({row["delta"]:+.2f})')

axes[0].set_xlabel('Selectivity delta = vina_Mut - vina_WT  (delta < 0: T790M-selective, delta > 0: WT-selective)')
axes[0].set_ylabel('Count')
axes[0].set_title('Selectivity Distribution by Pool')
axes[0].axvline(0, color='gray', linewidth=0.8, alpha=0.5)
axes[0].legend(fontsize=9)

# Right: cumulative distribution (CDF) comparison
for src, color, label in [
    ('WT_generated',  '#1f77b4', 'WT-generated'),
    ('Mut_generated', '#d62728', 'Mut-generated'),
]:
    sub = gen[gen['source'] == src]['delta'].sort_values()
    axes[1].plot(sub, np.linspace(0, 1, len(sub)), color=color, label=label, linewidth=2)

axes[1].axvline(0, color='gray', linewidth=0.8, alpha=0.5, label='equal binding')
axes[1].set_xlabel('Selectivity delta  (delta < 0: T790M-selective, delta > 0: WT-selective)')
axes[1].set_ylabel('Cumulative fraction')
axes[1].set_title('CDF Comparison (WT vs Mut pool)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/results/analysis/selectivity_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/analysis/selectivity_distribution.png')


---
## Section C — ADMET Profiling of Top Candidates
*CPU only · ~5 minutes*

Compare the top-5 Mut-selective candidates against the three approved EGFR drugs
across multiple drug-developability axes. We use RDKit-computable ADMET proxies
(all already present in `baselines.csv` / `filtered.csv`) plus a radar chart.

**ADMET proxies used:**
- **QED** (drug-likeness, 0–1)
- **SA score** (synthetic accessibility, 1=easy, 10=hard)
- **MW** (molecular weight, target 200–500 Da)
- **LogP** (lipophilicity, target 0–5)
- **TPSA** (topological polar surface area, <140 Å² for CNS; <160 Å² oral)
- **HBD / HBA** (hydrogen bond donors/acceptors — Lipinski indicators)


In [ ]:
# -- C1: Build ADMET comparison table (top-5 Mut-selective vs baselines) --
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, QED as QEDmod, rdMolDescriptors
from rdkit.Chem.rdMolDescriptors import CalcTPSA

from src.evaluation import compute_sa_score as sa_fn
def mol_props(smi, name=''):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return {
        'Name'  : name,
        'SMILES': smi,
        'MW'    : round(Descriptors.ExactMolWt(mol), 1),
        'LogP'  : round(Descriptors.MolLogP(mol), 2),
        'HBD'   : rdMolDescriptors.CalcNumHBD(mol),
        'HBA'   : rdMolDescriptors.CalcNumHBA(mol),
        'TPSA'  : round(CalcTPSA(mol), 1),
        'RotBonds': rdMolDescriptors.CalcNumRotatableBonds(mol),
        'QED'   : round(QEDmod.qed(mol), 3),
        'SA'    : sa_fn(mol),
    }

import yaml
cfg = yaml.safe_load(open(f'{PROJECT_ROOT}/configs/targets.yaml'))

rows = []
# Approved drugs
for drug, info in cfg['baselines'].items():
    rows.append(mol_props(info['smiles'], info['name']))

# Top-5 Mut-selective
top_mut = pd.read_csv(f'{PROJECT_ROOT}/results/analysis/top5_mut_selective.csv')
for idx, row in top_mut.iterrows():
    rows.append(mol_props(row['smiles'],
                          f'Candidate-{idx+1} (delta={row["delta"]:+.2f})'))

admet_df = pd.DataFrame([r for r in rows if r is not None])
print(admet_df.to_string(index=False))

admet_df.to_csv(f'{PROJECT_ROOT}/results/analysis/admet_comparison.csv', index=False)
print('\nSaved → results/analysis/admet_comparison.csv')


In [ ]:
# -- C2: Radar chart — top-3 Mut-selective candidates vs approved drugs --
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# admet_df is passed from C1 (already has SA column; do not reload from CSV here)
top_mut  = pd.read_csv(f'{PROJECT_ROOT}/results/analysis/top5_mut_selective.csv')
scores   = pd.read_csv(f'{PROJECT_ROOT}/results/vina_scores/combined_scores.csv')

# Normalise each axis to [0,1] so radar is comparable
# Axes: QED(higher=better), 1-SA/10(higher=better), 1-MW/600(closer to 300 ideal),
#       1-LogP/8(moderate), 1-TPSA/160(lower TPSA=better absorption), -vina_Mut/12
def normalise(row, scores_df):
    vina = scores_df[scores_df['smiles'] == row['SMILES']]['vina_Mut'].values
    vina_score = float(vina[0]) if len(vina) else -7.0
    return [
        row['QED'],                          # QED  (0-1)
        max(0, 1 - row['SA'] / 10),          # synthesizability
        max(0, 1 - abs(row['MW'] - 350) / 300),  # MW near 350 ideal
        max(0, 1 - abs(row['LogP'] - 3) / 5),    # LogP near 3 ideal
        max(0, 1 - row['TPSA'] / 160),        # TPSA
        min(1, abs(vina_score) / 12),          # binding strength
    ]

labels = ['QED', 'Synthesizability', 'MW (ideal ~350)', 'LogP (ideal ~3)', 'TPSA (absorption)', 'Binding Strength']
N = len(labels)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

colors_base = ['#2ca02c', '#ff7f0e', '#1f77b4']
colors_cand = ['#d62728', '#9467bd', '#8c564b']

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

# Plot approved drugs
drug_rows = admet_df[admet_df['Name'].isin(['Erlotinib', 'Gefitinib', 'Osimertinib'])]
for i, (_, row) in enumerate(drug_rows.iterrows()):
    vals = normalise(row, scores)
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, color=colors_base[i], label=row['Name'])
    ax.fill(angles, vals, alpha=0.07, color=colors_base[i])

# Plot top-3 candidates
cand_rows = admet_df[admet_df['Name'].str.startswith('Candidate')].head(3)
for i, (_, row) in enumerate(cand_rows.iterrows()):
    vals = normalise(row, scores)
    vals += vals[:1]
    ax.plot(angles, vals, 's--', linewidth=2, color=colors_cand[i],
            label=row['Name'][:22])
    ax.fill(angles, vals, alpha=0.07, color=colors_cand[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, size=11)
ax.set_ylim(0, 1)
ax.set_title('ADMET Radar: Top Candidates vs Approved Drugs', pad=20, fontsize=13)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)

plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/results/analysis/admet_radar.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/analysis/admet_radar.png')


---
## Section A — Pocket Radius Experiment (6 Å vs 10 Å)
*Requires GPU runtime (TargetDiff) · ~1.5 hours*

**Hypothesis:** The 10 Å pocket radius includes residues far from the T790M
mutation site (residue 790), diluting its structural signal. A tighter 6 Å radius
captures mainly the ATP-binding cleft where T790M directly influences geometry.

**Protocol:**
1. Re-extract 4I22 pocket at 6 Å → `4I22_pocket_6A.pdb`
2. Generate 20 molecules conditioned on this smaller pocket (TargetDiff)
3. Cross-dock these 20 molecules against both WT and T790M pockets
4. Compare selectivity Δ distribution: 6 Å cohort vs existing 10 Å cohort

**If selectivity signal increases** → confirms the radius hypothesis; paper-worthy.
**If similar** → pocket radius is not the limiting factor; model itself is the issue.


In [ ]:
# -- A1: Check GPU + env, then re-extract 4I22 pocket at 6 A --
import os, sys, subprocess
import torch

if not torch.cuda.is_available():
    print('WARNING: No GPU detected. Section A requires GPU for TargetDiff generation.')
    print('Switch runtime to GPU, re-run bootstrap, then run this section.')
else:
    print(f'GPU: {torch.cuda.get_device_name(0)}')

    # Re-extract 4I22 pocket at 6 A radius
    sys.path.insert(0, PROJECT_ROOT)
    from src.pocket_extraction import extract_pocket

    pdb_4i22    = f'{PROJECT_ROOT}/data/pdb/4I22.pdb'
    pocket_6a   = f'{PROJECT_ROOT}/data/pockets/4I22_pocket_6A.pdb'

    center = extract_pocket(
        pdb_file=pdb_4i22,
        ligand_resname='IRE',
        output_file=pocket_6a,
        radius=6.0,
        include_ligand=True,
    )
    print(f'\n6A pocket saved: {pocket_6a}')
    print(f'Center: {center.round(2)}')

    # Compare pocket sizes
    from Bio import PDB
    parser = PDB.PDBParser(QUIET=True)
    n_10a = sum(1 for _ in parser.get_structure('p', f'{PROJECT_ROOT}/data/pockets/4I22_pocket.pdb').get_residues())
    n_6a  = sum(1 for _ in parser.get_structure('p', pocket_6a).get_residues())
    print(f'\nResidue count: 10A pocket = {n_10a}  |  6A pocket = {n_6a}')


In [ ]:
# -- A2: Patch sampling config for 6A pocket and generate 20 molecules --
import os, subprocess, time, yaml

if not torch.cuda.is_available():
    print('Skip: no GPU.')
else:
    # Write a 6A-specific sampling config
    base_cfg_path = f'{PROJECT_ROOT}/configs/targetdiff_sampling.yml'
    with open(base_cfg_path) as f:
        cfg_6a = yaml.safe_load(f)
    cfg_6a['sample']['seed'] = 9999   # distinct seed from main runs

    cfg_6a_path = f'{PROJECT_ROOT}/configs/targetdiff_sampling_6A.yml'
    with open(cfg_6a_path, 'w') as f:
        yaml.dump(cfg_6a, f, sort_keys=False)

    POCKET_6A = f'{PROJECT_ROOT}/data/pockets/4I22_pocket_6A.pdb'
    OUT_DIR   = f'{PROJECT_ROOT}/results/generated/4I22_6A_smoke'
    SCRIPT    = os.path.join(TARGETDIFF, 'scripts', 'sample_for_pocket.py')
    os.makedirs(OUT_DIR, exist_ok=True)

    wrapper = (
        f"import sys, runpy; sys.path.insert(0, {TARGETDIFF!r}); "
        f"sys.argv = ['sample_for_pocket.py', {cfg_6a_path!r}, "
        f"'--pdb_path', {POCKET_6A!r}, "
        f"'--num_samples', '50', '--batch_size', '10', "
        f"'--result_path', {OUT_DIR!r}]; "
        f"runpy.run_path({SCRIPT!r}, run_name='__main__')"
    )
    env = os.environ.copy()
    env['PYTHONPATH'] = TARGETDIFF + os.pathsep + env.get('PYTHONPATH', '')

    print('Generating 20 mols with 6A 4I22 pocket ...')
    t0 = time.time()
    r = subprocess.run([CONDA_PY, '-c', wrapper],
                       cwd=TARGETDIFF, capture_output=True, text=True, env=env)
    print(f'Done in {time.time()-t0:.1f}s | rc={r.returncode}')
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1500:])
    else:
        import glob
        sdfs = glob.glob(f'{OUT_DIR}/sdf/*.sdf')
        print(f'Generated {len(sdfs)} SDFs')


In [ ]:
# -- A3: Cross-dock the 6A-generated molecules --
import glob, pandas as pd
from src.docking import VinaDocker, cross_dock_smiles_list
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
import json

if not torch.cuda.is_available():
    print('Skip: no GPU (generation not done).')
else:
    OUT_DIR_6A = f'{PROJECT_ROOT}/results/generated/4I22_6A_smoke'
    sdf_files  = sorted(glob.glob(f'{OUT_DIR_6A}/sdf/*.sdf'))
    if not sdf_files:
        print('No SDFs found — run A2 first.')
    else:
        # Load molecules
        smiles_6a = []
        for sdf in sdf_files:
            for m in Chem.SDMolSupplier(sdf, sanitize=True):
                if m is not None:
                    smiles_6a.append(Chem.MolToSmiles(m))
        smiles_6a = list(dict.fromkeys(smiles_6a))  # dedup
        print(f'Loaded {len(smiles_6a)} unique SMILES from 6A generation')

        # Build dockers (same receptors as nb07)
        with open(f'{PROJECT_ROOT}/data/pocket_centers.json') as f:
            centers = json.load(f)

        docker_wt  = VinaDocker(f'{PROJECT_ROOT}/data/pdb/1M17_receptor.pdbqt',
                                 tuple(centers['1M17']), exhaustiveness=8)
        docker_mut = VinaDocker(f'{PROJECT_ROOT}/data/pdb/4I22_receptor.pdbqt',
                                 tuple(centers['4I22']), exhaustiveness=8)

        results_6a = cross_dock_smiles_list(
            smiles_list=smiles_6a,
            docker_wt=docker_wt,
            docker_mut=docker_mut,
            output_csv=f'{PROJECT_ROOT}/results/vina_scores/4I22_6A_cross.csv',
            source_label='Mut_6A',
            checkpoint_every=5,
        )
        print(results_6a[['smiles','vina_WT','vina_Mut','status_WT','status_Mut']].head())


In [ ]:
# -- A4: Compare selectivity: 6A vs 10A cohort --
import pandas as pd, matplotlib.pyplot as plt
import os

path_6a = f'{PROJECT_ROOT}/results/vina_scores/4I22_6A_cross.csv'
path_10a = f'{PROJECT_ROOT}/results/vina_scores/combined_scores.csv'

if not os.path.exists(path_6a):
    print('6A docking results not found — run A2 and A3 first.')
else:
    df_6a  = pd.read_csv(path_6a)
    df_10a = pd.read_csv(path_10a)

    df_6a  = df_6a[(df_6a['status_WT']=='ok') & (df_6a['status_Mut']=='ok')].copy()
    df_10a = df_10a[df_10a['source']=='Mut_generated'].copy()

    df_6a['delta']  = df_6a['vina_Mut']  - df_6a['vina_WT']
    df_10a['delta'] = df_10a['vina_Mut'] - df_10a['vina_WT']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Histogram comparison
    axes[0].hist(df_10a['delta'], bins=15, alpha=0.6, color='#d62728',
                 label=f'10A pocket (n={len(df_10a)})')
    axes[0].hist(df_6a['delta'],  bins=15, alpha=0.6, color='#ff7f0e',
                 label=f'6A pocket  (n={len(df_6a)})')
    axes[0].axvline(0, color='gray', linewidth=0.8)
    axes[0].set_xlabel('Selectivity Δ = vina_Mut − vina_WT (kcal/mol)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Pocket Radius Effect on Selectivity Distribution')
    axes[0].legend()

    # Summary stats
    for df, label in [(df_10a, '10A'), (df_6a, '6A')]:
        print(f'{label}:  mean_delta={df["delta"].mean():.3f}  '
              f'std={df["delta"].std():.3f}  '
              f'Mut-selective(>0.5)={( df["delta"]>0.5).sum()}/{len(df)}')

    # Box plot
    axes[1].boxplot([df_10a['delta'], df_6a['delta']],
                    labels=['10A pocket', '6A pocket'],
                    patch_artist=True,
                    boxprops=dict(facecolor='#EBF5FB'),
                    medianprops=dict(color='#E74C3C', linewidth=2))
    axes[1].axhline(0, color='gray', linewidth=0.8, linestyle='--')
    axes[1].set_ylabel('Selectivity Δ (kcal/mol)')
    axes[1].set_title('Selectivity by Pocket Radius (Box Plot)')

    plt.tight_layout()
    plt.savefig(f'{PROJECT_ROOT}/results/analysis/pocket_radius_comparison.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → results/analysis/pocket_radius_comparison.png')


---
## Summary of Experiment Outputs

After running all three sections, the following files are written to
`results/analysis/`:

| File | Contents |
|------|---------|
| `top5_wt_selective.csv` | Top-5 WT-selective molecules with vina scores |
| `top5_mut_selective.csv` | Top-5 Mut-selective molecules with vina scores |
| `selectivity_distribution.png` | Delta histogram + CDF by pool (Experiment B) |
| `admet_comparison.csv` | ADMET table: top candidates vs approved drugs |
| `admet_radar.png` | Radar chart (Experiment C) |
| `pocket_radius_comparison.png` | 6A vs 10A selectivity comparison (Experiment A) |

**Next: Notebook 09** — polish the selectivity scatter, run KS-test, and
build the final narrative integrating all three experiments.
